**Data Ingestion**

In [3]:
from langchain_community.document_loaders import PyPDFLoader

/Users/ruchisehgal/Documents/Study Material/Industry Ready projects-LLMOPS/document_portal/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os
os.getcwd()

'/Users/ruchisehgal/Documents/Study Material/Industry Ready projects-LLMOPS/document_portal/notebook'

In [6]:
file_path=os.path.join(os.getcwd(),"data","sample.pdf")

In [7]:
loader=PyPDFLoader(file_path)
documents=loader.load()

In [8]:
len(documents)

77

Every document is a page here. Now below is chunking starting

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [44]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size= 500,
    chunk_overlap= 150,
    length_function=len
)

In [45]:
docs=text_splitter.split_documents(documents)
len(docs)

765

In [46]:
##so page_label is page number from which the data is coming
docs[0].page_content

'Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev'

In [47]:
docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/ruchisehgal/Documents/Study Material/Industry Ready projects-LLMOPS/document_portal/notebook/data/sample.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}, page_content='Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Vi

In [48]:
from langchain_community.vectorstores import FAISS
#we need to pass our embedding model on these documents and we will pass in from_documents and then will save in FAISS DB

In [20]:
from langchain_core.exceptions import ContextOverflowError


In [21]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector = embeddings.embed_query("hello, world!")
vector[:5]

[-0.023955047, 0.011876456, -0.0033613679, -0.0584139, 0.0015592978]

In [22]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


/var/folders/db/qwhv2d8s2ll6bgc2v1wtb4tw0000gn/T/ipykernel_87666/2236112934.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9475.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
vectorstore=FAISS.from_documents(docs, embeddings)

In [50]:
len(embeddings.embed_documents(docs[0].page_content))
# so size of this vector is 167

500

** Data Retrieval **

In [51]:
##Now we will perform a similarity search on this, This is called RETRIEVAL PROCESS

relevant_doc=vectorstore.similarity_search("What is the impact of data scaling") # put a comma and k=10/2(except 4 since its default to get those relevant result))
relevant_doc

[Document(id='3d0328d3-af97-4a4a-be0f-00136e3f0b8c', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/ruchisehgal/Documents/Study Material/Industry Ready projects-LLMOPS/document_portal/notebook/data/sample.pdf', 'total_pages': 77, 'page': 43, 'page_label': '44'}, page_content='Yuchen Hao, and Shen Li. Pytorch fsdp: Experiences on scaling fully sharded data parallel, 2023.\nWanjun Zhong, Ruixiang Cui, Yiduo Guo, Yaobo Liang, Shuai Lu, Yanlin Wang, Amin Saied, Weizhu Chen,\nand Nan Duan. Agieval: A human-centric benchmark for evaluating foundation models.arXiv preprint\narXiv:2304.06364, 2023.\nChunting Zhou, Pengfei Liu, Puxin Xu, Srini Iyer, Jiao Sun, Yuning Mao, Xuezhe Ma, 

In [52]:
relevant_doc[0].page_content

'Yuchen Hao, and Shen Li. Pytorch fsdp: Experiences on scaling fully sharded data parallel, 2023.\nWanjun Zhong, Ruixiang Cui, Yiduo Guo, Yaobo Liang, Shuai Lu, Yanlin Wang, Amin Saied, Weizhu Chen,\nand Nan Duan. Agieval: A human-centric benchmark for evaluating foundation models.arXiv preprint\narXiv:2304.06364, 2023.\nChunting Zhou, Pengfei Liu, Puxin Xu, Srini Iyer, Jiao Sun, Yuning Mao, Xuezhe Ma, Avia Efrat, Ping Yu, Lili'

In [53]:
relevant_doc

[Document(id='3d0328d3-af97-4a4a-be0f-00136e3f0b8c', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/ruchisehgal/Documents/Study Material/Industry Ready projects-LLMOPS/document_portal/notebook/data/sample.pdf', 'total_pages': 77, 'page': 43, 'page_label': '44'}, page_content='Yuchen Hao, and Shen Li. Pytorch fsdp: Experiences on scaling fully sharded data parallel, 2023.\nWanjun Zhong, Ruixiang Cui, Yiduo Guo, Yaobo Liang, Shuai Lu, Yanlin Wang, Amin Saied, Weizhu Chen,\nand Nan Duan. Agieval: A human-centric benchmark for evaluating foundation models.arXiv preprint\narXiv:2304.06364, 2023.\nChunting Zhou, Pengfei Liu, Puxin Xu, Srini Iyer, Jiao Sun, Yuning Mao, Xuezhe Ma, 

In [54]:
retriever=vectorstore.as_retriever() 
#method to do same thing as similarity serarch 
#in any line from that document to check relevant data is retrieved]
#We are calling to make it in chaining concept
retriever.invoke("What is the impact of data scaling")

[Document(id='3d0328d3-af97-4a4a-be0f-00136e3f0b8c', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/ruchisehgal/Documents/Study Material/Industry Ready projects-LLMOPS/document_portal/notebook/data/sample.pdf', 'total_pages': 77, 'page': 43, 'page_label': '44'}, page_content='Yuchen Hao, and Shen Li. Pytorch fsdp: Experiences on scaling fully sharded data parallel, 2023.\nWanjun Zhong, Ruixiang Cui, Yiduo Guo, Yaobo Liang, Shuai Lu, Yanlin Wang, Amin Saied, Weizhu Chen,\nand Nan Duan. Agieval: A human-centric benchmark for evaluating foundation models.arXiv preprint\narXiv:2304.06364, 2023.\nChunting Zhou, Pengfei Liu, Puxin Xu, Srini Iyer, Jiao Sun, Yuning Mao, Xuezhe Ma, 

Question: user question

Context: based on the question retrieving the info from the vector database

In [55]:
prompt_template= """
       Answe the question based on the context provided below.
       If the context doesnt contain the sufficient information, respond with:
       "I do not have enough information about this" 
       
       Context : {context}
       
       Question" {question}
       Answer: """

In [56]:
from langchain_core.prompts import PromptTemplate


In [57]:
#pip install -U langchain langchain-core langchain-community
# #pip install -U "langchain~=0.2.10" "langchain-core~=0.2.10"
prompt= PromptTemplate(
    template=prompt_template,
    input_variables=['context','question']
)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n       Answe the question based on the context provided below.\n       If the context doesnt contain the sufficient information, respond with:\n       "I do not have enough information about this" \n\n       Context : {context}\n\n       Question" {question}\n       Answer: ')

In [58]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b"
)

print(llm.invoke("What is the capital of France?").content)

<think>
Okay, so I need to figure out what the capital of France is. Let me start by recalling any information I have about France. France is a country in Europe, right? I remember learning about European capitals in school. Some countries have capitals like Berlin for Germany, Rome for Italy, Madrid for Spain. So France... maybe Paris? I think Paris is a big city in France. I know it's famous for the Eiffel Tower and the Louvre Museum. But wait, is it the capital?

Wait, could it be Lyon? I've heard Lyon mentioned in some contexts, but I'm not sure about its role. Or maybe Marseille? Marseille is a port city, I think. But I'm pretty sure Paris is more central. Let me try to remember: the government of France is based in the capital. The president's office, the parliament, those institutions should be in the capital. I believe the President of France resides in the Élysée Palace, which I think is in Paris. That would make sense if Paris is the capital.

Another angle: I've heard of Par

In [43]:
#Now we have to create the RAG chain WITH Chaining concept
#StrOutputParser convert the output into string
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
rag_chain=prompt | llm | parser

In [63]:
#Inside the chain, we want context from receiver and user question 
#and format_docs will provide the PAGE CONTENT only and RunnablePassthrough() is used to get the question on run time.
#format_docs --FUNCTION NAME NAME IS GETTING CREATED
from langchain_core.runnables import RunnablePassthrough
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [66]:
rag_chain=(

    {"context":retriever | format_docs, "question": RunnablePassthrough()}

    | prompt | llm | StrOutputParser()
)

#CAN USE Runnablelambda(format_docs) to convert to text

In [67]:

rag_chain.invoke("What is the impact of data scaling")

"<think>\nOkay, let's see. The user is asking about the impact of data scaling. The context provided mentions a study where they looked at how adding safety training data affects the model's performance, especially in terms of helpfulness. The key point there is that even after adding more safety data during the RLHF stage, the model's helpfulness doesn't degrade notably. They also reference a study by Bai et al. (2022a) about the tension between helpfulness and safety in LLMs.\n\nSo, the impact of scaling safety data, according to the context, is that it doesn't harm the model's helpfulness. The example given in Table 12 shows a qualitative example supporting this. The main idea is that with enough helpfulness training data, adding safety measures doesn't negatively affect performance. \n\nI need to make sure I'm not missing other parts of the context. The other parts mention PyTorch FSDP, Agieval benchmark, and scaling laws for neural models, but those don't directly relate to the qu